# MetaRCWA Square Particle

This notebook constructs and solves one periodic square particle structure using Metarcwa.


In [ ]:
import torch
import yaml

from dataclasses import dataclass
from dataclasses import replace
from pathlib import Path

# Make newly created floating-point tensors use float64 by default.
torch.set_default_dtype(torch.float64)

from dispertorch import ConstantEps

from metashapes.shape import Rectangle

from metarcwa import (
    Config as MetaRCWASolverConfig,
    Lattice,
    Layer,
    IsotropicMedium,
    Model, 
    Solver, 
    Factorization, 
    Observables,
    Source,
    Stack)

from metarcwa.solver.harmonics import harmonic_index_map

from metarcwa.model.adapters import (
    from_dispertorch,
    from_metashapes
)

In [2]:
@dataclass
class SquareParticleModel:
    """Physical description of one periodic square particle structure.

    All lengths are expressed in nanometers. Angles are expressed in degrees because
    that is what S4 expects. 

    Attributes
    ----------
    period_nm: 
        Period of the square lattice in both x and y
    particle_side_nm:
        Full side length of the square particle.
    planarization_thickness_nm:
        Thickness of the homogeneous planarization layer.
    patterned_layer_thickness_nm:
        Thickness of the layer containing the square particle. 
    wavelength_nm:
        Free-space wavelength.
    theta_deg:
        Polar incidence angle measured from the surface normal.
    phi_deg:
        Azimuthal incidence angle.
    incidence_epsilon:
        Permittivity of the semi-infinite incidence material
    planarization_epsilon:
        Permittivity of the homogeneous planarization layer
    background_epsilon:
        Permittivity of the material surrounding the square particle
    particle_epsilon:
        Permittivity of the square particle.
    transsmission_epsilon:
        Permittivity of the semi-infinite transmission material
    square_corner_radius:
        
    """

    period_nm: float
    particle_side_nm: float
    planarization_thickness_nm: float
    patterned_layer_thickness_nm: float
    wavelength_nm: float
    theta_deg: float
    phi_deg: float
    incidence_epsilon: complex
    planarization_epsilon: complex
    background_epsilon: complex
    particle_epsilon: complex
    transmission_epsilon: complex
    square_corner_radius: float

    @classmethod
    def from_dict(cls, data:dict) -> "SquareParticleModel":
        """Create a physical model from YAML-derived dictionary data."""

        # Copy the dictionary so we do not modify the original data
        data = dict(data)

        epsilon_list = (
            "incidence_epsilon",
            "planarization_epsilon",
            "background_epsilon", 
            "particle_epsilon",
            "transmission_epsilon",
        )

        # Convert each {real:..., image:...} dictionary into 
        # an ordinary Python complex number
        for epsilon in epsilon_list:
            components = data[epsilon]

            data[epsilon] = complex(
                components["real"],
                components.get("imag",0.0)
            )

        return cls(**data)

In [3]:
@dataclass
class MetaRCWAConfig:
    """Numerical settings used directly by MetaRCWA

    Attributes
    ----------
    dtype: str or torch.dtype
        Data type used for the floating point precision
    device: str or torch.device
        Hardware on which the calculations are performed
        'cpu' or 'cuda'
    nx: int
        Number of real-space sampling points along the 
        a1 lattice vector
    ny: int
        The grid resolution along a2 lattice vector
    m: int
        Maximum Fourier index retained in the a1 direction,
        which gives 2m+1 harmonics altogether along that axis
    n: int
        Maximum Fourier index retained in the a2 direction, 
        giving 2n+1 harmonics along that axis.
    truncation: str
        The rule used to select the Fourier harmonics retained.
        circular selects modes within an elliptical boundary and 
        rectangular retains the complex (2m+1) * (2n+1). 
    """

    dtype: str
    device: str
    nx: int
    ny: int
    m: int
    n: int
    truncation: str
    

    def __post_init__(self) -> None:
        """Check that the supplied settings are valid."""

        if self.nx <=0 or self.ny <=0:
            raise ValueError("nx and ny must be positive")

        if self.m <=0 or self.n<=0:
            raise ValueError("n and m cannot be negative")

        allowed_truncations = {
            "circular",
            "rectangular"
        }

        if self.truncation not in allowed_truncations:
            raise ValueError(
                "truncation must be "
                "'circular' or 'rectangular'"
            )

    @classmethod
    def from_dict(cls, data:dict) -> "MetaRCWAConfig":
        "Create the numerical settings from YAML derived dictionary data."

        # Copy the dictionary so you don't modify the original data
        data = dict(data)

        return cls(**data)

In [4]:
config_path = Path("../src/configs/square_particle.yaml")

with config_path.open("r") as file:
    yaml_data = yaml.safe_load(file)

In [5]:
model = SquareParticleModel.from_dict(
    yaml_data["model"]
)

config = MetaRCWAConfig.from_dict(
    yaml_data["metarcwa"]
)

In [6]:
def build_metarcwa_model(
        model: SquareParticleModel
) -> Model:
    """Create a square particle model for MetaRCWA.
    """ 
    # Convert the scalar quantities into PyTorch Tensors

    lattice_period_nm= torch.tensor(
        model.period_nm
    )

    particle_side_length_nm = torch.tensor(
        model.particle_side_nm)

    planarization_thickness_nm = torch.tensor(
        model.planarization_thickness_nm
    )

    patterned_layer_thickness_nm = torch.tensor(
        model.patterned_layer_thickness_nm
    )

    # MetaRCWA treats wavelength, theta and phi as independent
    # sweep axes so each is a length one tensor
    wavelength_nm = torch.tensor([
        model.wavelength_nm
    ])

    theta_rad = torch.deg2rad(
        torch.tensor(
        [model.theta_deg]
    )
    )

    phi_rad = torch.deg2rad(
        torch.tensor(
            [model.phi_deg]
        )
    )

    # Define the square lattice primitive vectors
    lattice = Lattice.rectangular(
        px=lattice_period_nm,
        py=lattice_period_nm
    )

    # ConstantEps(eps_re, eps_im)
    # Even though we called default dtype float64 
    # at the top, ConstantEps defines its own constructors 
    # where it defaults to float32 so will re-specify
    incidence_perm_model = ConstantEps(
        eps_re = model.incidence_epsilon.real,
        eps_im = model.incidence_epsilon.imag,
        dtype=torch.float64
        )

    planarization_perm_model = ConstantEps(
        eps_re = model.planarization_epsilon.real,
        eps_im = model.planarization_epsilon.imag,
        dtype=torch.float32
        )

    particle_perm_model = ConstantEps(
        eps_re = model.particle_epsilon.real,
        eps_im = model.particle_epsilon.imag,
        dtype=torch.float32
        )

    pattern_background_perm_model = ConstantEps(
        eps_re = model.background_epsilon.real,
        eps_im = model.background_epsilon.imag,
        dtype=torch.float32
    )

    transmission_perm_model = ConstantEps(
        eps_re = model.transmission_epsilon.real,
        eps_im = model.transmission_epsilon.imag,
        dtype=torch.float32
    )

    # Convert the DisperTorch Material Models into 
    # MetaRCWA media

    incidence_medium = IsotropicMedium(
        from_dispertorch(incidence_perm_model)
    )

    planarization_medium = IsotropicMedium(
        from_dispertorch(planarization_perm_model)
    )

    particle_medium = IsotropicMedium(
        from_dispertorch(particle_perm_model)    
        )

    pattern_background_medium = IsotropicMedium(
        from_dispertorch(pattern_background_perm_model)
    )

    transmission_medium = IsotropicMedium(
        from_dispertorch(transmission_perm_model)
    )

    # MetaShapes uses coordinates within [0, period]
    square_center = torch.tensor([lattice_period_nm/2, 
                                  lattice_period_nm/2])
    size = torch.tensor([particle_side_length_nm,
                         particle_side_length_nm])
    angle = torch.tensor(0.0)
    corner_radius = torch.tensor(model.square_corner_radius)
    square_geometry = Rectangle(center=square_center,
                                size=size,
                                angle=angle,
                                corner_radius=corner_radius)

    # Define finite layers
    planarization_layer = Layer(medium_solid=planarization_medium,
                                thickness = planarization_thickness_nm
                                )
    patterned_layer = Layer(medium_solid = particle_medium,
                            thickness = patterned_layer_thickness_nm,
                            medium_void = pattern_background_medium,
                            shape_fn = from_metashapes(square_geometry))

    # Stack the layers
    stack = Stack(incidence = incidence_medium,
                  layers = [planarization_layer, patterned_layer],
                  transmission = transmission_medium,
                  lattice = lattice
                  )

    source = Source(wavelength = wavelength_nm,
                    theta = theta_rad,
                    phi = phi_rad
                    )

    return Model(stack, source)

In [19]:
def build_metarcwa_config(
    config: MetaRCWAConfig
) -> MetaRCWASolverConfig:
    """
    Create Config for MetaRCWA
    """
    dtype_map = {
        "float32": torch.float32,
        "float64": torch.float64
    }

    return MetaRCWASolverConfig(
        dtype=dtype_map[config.dtype],
        device = config.device,
        nx = config.nx,
        ny=config.ny,
        m=config.m,
        n=config.n,
        truncation=config.truncation,
        factorization=None
    )

In [ ]:
def reflectance_and_transmittance_metarcwa(
        model:Model,
        config:MetaRCWASolverConfig
):
    solver = Solver(model,config)
    solution = solver.run()
    obs = Observables(solution)
    # Compute optical qunatities
    # Compute optical qunatities
    rs, _ = obs.reflection(pol='s')
    _, rp = obs.reflection(pol='p')
    Rs = torch.abs(rs.cpu())**2
    Rp = torch.abs(rp.cpu())**2

    ts, _ = obs.transmission(pol='s')
    _, tp = obs.transmission(pol='p')
    
    n1 = (model.stack.incidence.eps_fn(model.source.wavelength)**0.5).cpu().reshape(-1, 1, 1)
    n2 = (model.stack.transmission.eps_fn(model.source.wavelength)**0.5).cpu().reshape(-1, 1, 1)

    theta = model.source.theta.cpu().reshape(1, -1, 1)
    cos_i = torch.cos(theta)
    sin_t = (n1 / n2) * torch.sin(theta)
    cos_t = torch.sqrt(1 - sin_t**2 + 0j)
    coeff = torch.real(n2 * cos_t) / torch.real(n1 * cos_i)
    
    Ts = coeff * torch.abs(ts.cpu())**2
    Tp = coeff * torch.abs(tp.cpu())**2
    return Rs, Rp, Ts, Tp

In [ ]:
# range() excludes the final value, so use 101 to include 100
requested_extents = [1,2,3,4]

print(requested_extents)

metarcwa_rs_values = []
metarcwa_actual_orders = []
metarcwa_requested_orders=[]

for extent in requested_extents:

    sweep_config = replace(
        config,
        m=extent,
        n=extent
    )

    # Rebuild config because the Fourier orders have changed
    sweep_solver_config = build_metarcwa_config(
        config=sweep_config
    )

    # Build the model
    metarcwa_physical_model = build_metarcwa_model(model,
                                                   config=sweep_config)

    Rs, Rp, Ts, Tp = (
        reflectance_and_transmittance_metarcwa(
            model=metarcwa_physical_model,
            config=sweep_solver_config
        )
    )

    # squeeze() to remove dimensions when length is one
    

    # Store this simulation's results
    metarcwa_rs_values.append(rs)
    actual_num_basis = 2*requested_m + 1 
    metarcwa_actual_orders.append(actual_num_basis)

    print(
        f"Requested: {:3d}, "
        f"actual: {actual_num_basis:3d}, "
        f"Rs: {Rs:.10f}"
    )

TypeError: '<=' not supported between instances of 'list' and 'int'